In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os


# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras
from keras.preprocessing import image
from keras.models import Sequential
from keras.layers import Conv2D, MaxPool2D, Flatten,Dense,Dropout,BatchNormalization,GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import cv2
from tensorflow.keras.applications import EfficientNetB2
#from tensorflow.keras.applications import EfficientNetB4
from keras import regularizers
from tensorflow.keras.optimizers import Adam,RMSprop,SGD,Adamax
from tensorflow.keras.applications.efficientnet import preprocess_input

1. Define directories                    
2. Check class distribution              
3. Load data with image_dataset_from_directory  
4. Compute class weights                 
5. Define augmentation layers            
6. Build model (augmentation goes inside)
7. Compile + train

**Defining directories and creating the datasets**

In [ ]:
train_dir='/kaggle/input/datasets/msambare/fer2013/train'
test_dir='/kaggle/input/datasets/msambare/fer2013/test'

for emotion in sorted(os.listdir(train_dir)):
    path=os.path.join(train_dir,emotion)
    print(emotion,len(os.listdir(path)))

**Note**: image_dataset_from_directory  
\
main_dataset_directory/\
    ├── class_a_cats/\
    │     &emsp;├── cat_001.jpg\
    │     &emsp;└── cat_002.png\
    └── class_b_dogs/\
          &emsp;├── dog_001.jpg\
          &emsp;└── dog_002.png\
For image_dataset_from_directory to work automatically, your folder structure must be grouped into subdirectories named after your class labels. The function uses the folder names to infer the target labels.

In [ ]:
img_size=96 #size expected by efficientnetb2: 96, and efficientnetb4: 128


train_ds=tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(img_size,img_size),
    color_mode='rgb', #greyscale to rgb conversion
    batch_size=32,
    #label_mode='int', #0->angry, 1->disgust etc ; can be used w sparse_categorical_crossentropy-> efficientnetB2
    label_mode='categorical',
    shuffle=True,
    seed=42 #easy to shuffle
)

test_ds=tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(img_size,img_size),
    color_mode='rgb',
    batch_size=32,
    label_mode='categorical',
    shuffle=False,
)

In [ ]:
labels = np.concatenate([y for x, y in train_ds], axis=0)
labels_int = np.argmax(labels, axis=1)
print(np.unique(labels_int, return_counts=True))

**Computing class weights**

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
labels=np.concatenate([y for x,y in train_ds],axis=0)
class_weights=compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels_int),
    y=labels_int
)
class_weight_dict=dict(enumerate(class_weights))
print(class_weight_dict)

**Data Augmentation**

In [ ]:
data_augmentation=tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.05),
    tf.keras.layers.RandomTranslation(0.05, 0.05),
    tf.keras.layers.RandomContrast(0.1),
], name="augmentation")

**PHASE 1: Frozen base, train head only (15 epochs)** 
<br>
EfficientNetB2 starts with ImageNet weights — it already knows how to detect edges, textures, shapes. You freeze all of that and only train your custom Dense layers on top. <br>
Why: if you unfreeze everything immediately, the random head weights produce chaotic gradients that destroy the pretrained weights before they can be useful.

In [ ]:
base_model=EfficientNetB2(
    include_top=False, 
    input_shape=(96,96,3),
    weights='imagenet'
)
base_model.trainable=False

inputs=tf.keras.Input(shape=(96,96,3))
x=data_augmentation(inputs)
x=preprocess_input(x)
x=base_model(x, training=False)
x=GlobalAveragePooling2D()(x) #instead of flatten 
x = BatchNormalization()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(7, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau,ModelCheckpoint
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    #loss='sparse_categorical_crossentropy',
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)
callbacks=[
    EarlyStopping(monitor='val_accuracy', patience=5,restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
    ModelCheckpoint('best_model_phase1.keras', save_best_only=True)
]
history1=model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=15,
    class_weight=class_weight_dict, #computed class weights 
    callbacks=callbacks
)

In [ ]:
model.save('phase1_model.keras')

**PHASE 2: FINE TUNING- Unfreeze top 40 layers (30 epochs)** <BR>
Now the head knows roughly what it's doing, so you can start nudging the top layers of EfficientNetB4. Lower lr (1e-4) so you make small adjustments, not big destructive ones.<BR>
Why top 40 only: early layers detect generic things like edges and corners — those are useful for every vision task, don't touch them. Later layers detect higher-level features like shapes and textures — those need fine-tuning for faces specifically.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau,ModelCheckpoint
#unfreezes top 40 efficientnet layers 
base_model.trainable=True
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
for layer in base_model.layers[:-40]:
    layer.trainable=False

#recompile w much lower LR
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)
callbacks_phase2=[
    EarlyStopping(monitor='val_accuracy', patience=7,restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
    ModelCheckpoint('best_model_phase2.keras', save_best_only=True)
]
history2 = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=30,
    class_weight=class_weight_dict,
    callbacks=callbacks_phase2
)

In [ ]:
model.save('phase2_done.keras')

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

y_pred = np.argmax(model.predict(test_ds), axis=1)
y_true = np.concatenate([y for x, y in test_ds], axis=0)
y_true = np.argmax(y_true, axis=1)

print(classification_report(y_true, y_pred, 
      target_names=['angry','disgust','fear','happy','neutral','sad','surprise']))

**PHASE 3:FINE TUNING FURTHER- UNFREEZE TOP 120 LAYERS(20 epochs)**

In [ ]:
base_model.trainable = True

for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

for layer in base_model.layers[:-120]:
    layer.trainable = False
    
model.compile(optimizer=Adam(1e-5), loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1), metrics=['accuracy'])
cb3 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
    tf.keras.callbacks.ModelCheckpoint('best_phase3.keras', save_best_only=True)
]
model.fit(train_ds, validation_data=test_ds, epochs=20, class_weight=class_weight_dict, callbacks=cb3)
model.save('phase3_done.keras')

**Classification Report**

In [ ]:
import tensorflow as tf
model = tf.keras.models.load_model('/kaggle/input/notebooks/medhaaaapaulo/fer-one-block/best_phase3.keras')

import numpy as np
from sklearn.metrics import classification_report
test_dir  = '/kaggle/input/datasets/msambare/fer2013/test'
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(96, 96),
    color_mode='rgb',
    batch_size=32,
    label_mode='categorical',
    shuffle=False
)

y_pred = np.argmax(model.predict(test_ds), axis=1)
y_true = np.argmax(np.concatenate([y for x, y in test_ds], axis=0), axis=1)

print(classification_report(y_true, y_pred,
      target_names=['angry','disgust','fear','happy','neutral','sad','surprise']))

**RESULTS**
Training accuracy: 76.65%  <br>
Validation accuracy: 63.58% <br>
Model: EfficientNetB2 (Transfer Learning)
Dataset: FER2013

**INFERENCE**

The EfficientNetB2-based facial emotion recognition model achieved a training accuracy of 76.65% and a validation accuracy of 63.58% on the FER2013 dataset. The gap between training and validation accuracy indicates mild overfitting, which is expected due to the dataset's limited size, class imbalance, and noisy labels. <br>
Future improvements could include higher-resolution training, test-time augmentation (TTA), or experimenting with alternative backbones and larger facial expression datasets to further improve performance.